# DINOv3 report rerun: complete selection and per-image evidence

Select a **Colab GPU kernel**, then **Run All** and authorize Google Drive. The original completed experiment used an A100; this notebook also supports T4 using FP16 and a smaller embedding batch. Supply `HF_TOKEN` through the environment or Colab Secrets, or enter it at the hidden prompt. The token must read the dataset and the gated DINOv3 backbone.

This reproduces the original **84 training runs** (2 training plans × 7 learning rates × 2 weight decays × 3 seeds), with a frozen DINOv3 ViT-B/16 backbone, CLS pooling, and a two-logit linear classifier. Each classifier has both a hazard output and a hazard-subtype output.

Two sets of test results are saved with explicit labels:
- `reported_A` and `reported_B`: fixed configurations from the original report's final test-summary columns, to reproduce the reported rows.
- `validation_*`: choices recomputed from this run's validation results using the original selection rule: select learning rate by mean precision@46 within each training-plan/weight-decay group, then each threshold by its own balanced accuracy.

Stable configuration IDs and explicit identity checks prevent old numeric run indices from referring to different models. The historical report configurations are **fixed reproduction targets**, not claimed to be this rerun's validation winners. Test results never feed selection.

Everything is saved under `MyDrive/MLiP/dino_report_reruns/<run-id>/`. Completed training trials are saved individually. To resume after a disconnect, set `RESUME_RUN_DIR` below to that Drive folder and Run All; completed trials are reused and an interrupted trial restarts. Do not change settings while resuming.

The original A100 sweep recorded about 19 minutes of head training, plus setup/feature extraction. This is a historical observation, not a T4 runtime guarantee. No model has been run by the assistant while preparing this notebook.

In [ ]:
%pip install -q "transformers==5.22.1" pandas scikit-learn matplotlib huggingface_hub
import torch, transformers
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running this notebook.'
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## Configuration
The training settings match the completed experiment. All reported thresholds are evaluated after training; they do not multiply the number of training runs. `RUN_TEST=True` runs the fixed report configurations and the fresh validation choices at the end.

In [ ]:
from pathlib import Path

# Data
DATA_REPO = "gvadhul/sidewalk-proc-A-naive-merged-nulls"                      # private Hugging Face dataset repo holding DATA_TAR, e.g. "user/sidewalk-proc"
DATA_TAR = "proc-sidewalk-data-naive-merged-nulls.tar"
DATA_ROOT = Path("/content/proc-sidewalk-data-naive-merged-nulls")
CACHE_BATCH_SIZE = 256 if torch.cuda.get_device_properties(0).total_memory >= 30 * 2**30 else 64

# Labels: decided here, not in the processed dataset
PROBLEM_CLASSES = ["Obstacle", "SurfaceProblem"]
CLEAN_CLASSES = ["Crosswalk", "CurbRamp", "no_obstacles", "no_obstacles_manual_collection"]
EVAL_SPLIT = "pitt-val"
TEST_SPLIT = "pitt-test"
EVAL_DROP = []
K = 46                                # stage 1 Recall@K
THRESHOLD = 0.5                       # used during training; the results below use THRESHOLDS
THRESHOLDS = [0.3, 0.5, 0.7]          # stage 1 and stage 2 grid

TRAIN_PLANS = {
    "pitt": [("pitt-train", 5000)],
    "other_then_pitt": [("other-cities-train", 5000), ("pitt-train", 5000)],
}

BASE = dict(
    base_model="facebook/dinov3-vitb16-pretrain-lvd1689m",
    pooling="cls",                    # "cls", "mean" (of patch tokens) or "cls+mean"
    train="pitt",                     # a TRAIN_PLANS key
    lr=1e-4,
    batch_size=64,
    eval_every=250,
    seed=0,
)

SWEEPS = [
    dict(train=["pitt", "other_then_pitt"], lr=[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2], weight_decay = [0.0, 1e-2], seed=[0, 1, 2]),
]
DRIVE_ROOT = Path('/content/drive/MyDrive/MLiP/dino_report_reruns')
RESUME_RUN_DIR = None  # e.g. '/content/drive/MyDrive/MLiP/dino_report_reruns/20261117T...'
RUN_TEST = True
REPORTED_CHOICES = [
    dict(choice_id='reported_A', selection_kind='historical_report_reproduction', train='pitt', lr=1e-5, weight_decay=0.0, thresh1=0.3, thresh2=0.3),
    dict(choice_id='reported_B', selection_kind='historical_report_reproduction', train='pitt', lr=1e-4, weight_decay=1e-2, thresh1=0.7, thresh2=0.3),
]
EXPECTED_MANIFEST_SHA256 = {'other-cities-train': '57e004a3b6d9e5f0482be285f8bea920b371428caf4fe9ab6c44c13ef9af19a2', 'pitt-train': '081723b38ebf0b5448d56e67be6b0c99dafdf2e3cf239e7d7a239b5fcfaad0cf', 'pitt-val': 'ae5cc3b382794050c8ad92dcb28ea1a4b82240ed266d329404a67e791a5e69de', 'pitt-test': '6dc4e536d46a57ebb0d3ca78175d7429a4aed6513a93877ec0ae413562dbb818'}

In [ ]:
from itertools import product

def expand(sweeps, base):
    return [{**base, **dict(zip(s, values))} for s in sweeps for values in product(*s.values())]

CONFIGS = expand(SWEEPS, BASE)
GROUP = list(dict.fromkeys(k for s in SWEEPS for k in s if k != "seed"))  # what results average seeds over
COMPARE = [k for k in GROUP if k != "lr"]                                  # each of these gets its own lr and thresholds
print(len(CONFIGS), "runs")
import hashlib, json
def trial_id(cfg):
    return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]
assert len(CONFIGS) == 84 and len({trial_id(c) for c in CONFIGS}) == 84
assert set(c['seed'] for c in CONFIGS) == {0, 1, 2}

## Persistent Drive output
This cell must finish before data loading or training. The run configuration and notebook computation code are saved with the results.

In [ ]:
import os, sys, time, tempfile, shutil, importlib.metadata
from datetime import datetime, timezone
from uuid import uuid4
from google.colab import drive
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive did not mount.'

def jsonable(x):
    if hasattr(x, 'tolist'): return x.tolist()
    if isinstance(x, Path): return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name('.' + path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=jsonable) + '\n')
    os.replace(temporary, path)

def save_torch(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='dino-save-') as temporary:
        local = Path(temporary) / 'checkpoint.pt'
        torch.save(value, local)
        pending = path.with_name('.' + path.name + '.tmp')
        shutil.copyfile(local, pending)
        os.replace(pending, path)

EXPERIMENT_SOURCE = '# CELL 1\n%pip install -q "transformers==5.22.1" pandas scikit-learn matplotlib huggingface_hub\nimport torch, transformers\nassert torch.cuda.is_available(), \'Select a Colab GPU runtime before running this notebook.\'\nprint(\'torch\', torch.__version__, \'| transformers\', transformers.__version__)\nprint(\'GPU:\', torch.cuda.get_device_name(0))\n\n# CELL 3\nfrom pathlib import Path\n\n# Data\nDATA_REPO = "gvadhul/sidewalk-proc-A-naive-merged-nulls"                      # private Hugging Face dataset repo holding DATA_TAR, e.g. "user/sidewalk-proc"\nDATA_TAR = "proc-sidewalk-data-naive-merged-nulls.tar"\nDATA_ROOT = Path("/content/proc-sidewalk-data-naive-merged-nulls")\nCACHE_BATCH_SIZE = 256 if torch.cuda.get_device_properties(0).total_memory >= 30 * 2**30 else 64\n\n# Labels: decided here, not in the processed dataset\nPROBLEM_CLASSES = ["Obstacle", "SurfaceProblem"]\nCLEAN_CLASSES = ["Crosswalk", "CurbRamp", "no_obstacles", "no_obstacles_manual_collection"]\nEVAL_SPLIT = "pitt-val"\nTEST_SPLIT = "pitt-test"\nEVAL_DROP = []\nK = 46                                # stage 1 Recall@K\nTHRESHOLD = 0.5                       # used during training; the results below use THRESHOLDS\nTHRESHOLDS = [0.3, 0.5, 0.7]          # stage 1 and stage 2 grid\n\nTRAIN_PLANS = {\n    "pitt": [("pitt-train", 5000)],\n    "other_then_pitt": [("other-cities-train", 5000), ("pitt-train", 5000)],\n}\n\nBASE = dict(\n    base_model="facebook/dinov3-vitb16-pretrain-lvd1689m",\n    pooling="cls",                    # "cls", "mean" (of patch tokens) or "cls+mean"\n    train="pitt",                     # a TRAIN_PLANS key\n    lr=1e-4,\n    batch_size=64,\n    eval_every=250,\n    seed=0,\n)\n\nSWEEPS = [\n    dict(train=["pitt", "other_then_pitt"], lr=[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2], weight_decay = [0.0, 1e-2], seed=[0, 1, 2]),\n]\nDRIVE_ROOT = Path(\'/content/drive/MyDrive/MLiP/dino_report_reruns\')\nRESUME_RUN_DIR = None  # e.g. \'/content/drive/MyDrive/MLiP/dino_report_reruns/20261117T...\'\nRUN_TEST = True\nREPORTED_CHOICES = [\n    dict(choice_id=\'reported_A\', selection_kind=\'historical_report_reproduction\', train=\'pitt\', lr=1e-5, weight_decay=0.0, thresh1=0.3, thresh2=0.3),\n    dict(choice_id=\'reported_B\', selection_kind=\'historical_report_reproduction\', train=\'pitt\', lr=1e-4, weight_decay=1e-2, thresh1=0.7, thresh2=0.3),\n]\nEXPECTED_MANIFEST_SHA256 = {\'other-cities-train\': \'57e004a3b6d9e5f0482be285f8bea920b371428caf4fe9ab6c44c13ef9af19a2\', \'pitt-train\': \'081723b38ebf0b5448d56e67be6b0c99dafdf2e3cf239e7d7a239b5fcfaad0cf\', \'pitt-val\': \'ae5cc3b382794050c8ad92dcb28ea1a4b82240ed266d329404a67e791a5e69de\', \'pitt-test\': \'6dc4e536d46a57ebb0d3ca78175d7429a4aed6513a93877ec0ae413562dbb818\'}\n\n# CELL 4\nfrom itertools import product\n\ndef expand(sweeps, base):\n    return [{**base, **dict(zip(s, values))} for s in sweeps for values in product(*s.values())]\n\nCONFIGS = expand(SWEEPS, BASE)\nGROUP = list(dict.fromkeys(k for s in SWEEPS for k in s if k != "seed"))  # what results average seeds over\nCOMPARE = [k for k in GROUP if k != "lr"]                                  # each of these gets its own lr and thresholds\nprint(len(CONFIGS), "runs")\nimport hashlib, json\ndef trial_id(cfg):\n    return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]\nassert len(CONFIGS) == 84 and len({trial_id(c) for c in CONFIGS}) == 84\nassert set(c[\'seed\'] for c in CONFIGS) == {0, 1, 2}\n\n# CELL 8\nimport tarfile, time\nfrom concurrent.futures import ThreadPoolExecutor\nimport numpy as np, pandas as pd\nfrom huggingface_hub import hf_hub_download\nfrom PIL import Image\n\nimport getpass\nfrom huggingface_hub import HfApi\nHF_TOKEN = os.environ.get(\'HF_TOKEN\', \'\').strip()\nif not HF_TOKEN:\n    try:\n        from google.colab import userdata\n        HF_TOKEN = (userdata.get(\'HF_TOKEN\') or \'\').strip()\n    except Exception:\n        HF_TOKEN = \'\'\nif not HF_TOKEN:\n    HF_TOKEN = getpass.getpass(\'Hugging Face read token (dataset and DINOv3 access): \').strip()\nif not HF_TOKEN:\n    raise RuntimeError(\'HF_TOKEN is required for the private dataset and gated DINOv3 model.\')\n\nif not all((DATA_ROOT / f\'{s}.csv\').is_file() for s in EXPECTED_MANIFEST_SHA256):\n    t0 = time.perf_counter()\n    revision = HfApi(token=HF_TOKEN).dataset_info(DATA_REPO).sha\n    tar = hf_hub_download(DATA_REPO, DATA_TAR, repo_type=\'dataset\', revision=revision, token=HF_TOKEN)\n    with tarfile.open(tar) as archive:\n        archive.extractall(DATA_ROOT.parent, filter=\'data\')\n    save_json(RUN_DIR / \'download.json\', dict(repo=DATA_REPO, revision=revision, archive=DATA_TAR,\n        archive_sha256=hashlib.file_digest(open(tar, \'rb\'), \'sha256\').hexdigest()))\n    print(f\'downloaded and extracted in {time.perf_counter() - t0:.0f}s\')\n\nfor split, expected in EXPECTED_MANIFEST_SHA256.items():\n    manifest = DATA_ROOT / f\'{split}.csv\'\n    assert hashlib.sha256(manifest.read_bytes()).hexdigest() == expected, f\'Dataset drift: {split}.csv does not match the report dataset.\'\n    snapshot = RUN_DIR / \'manifests\' / manifest.name\n    snapshot.parent.mkdir(exist_ok=True)\n    shutil.copy2(manifest, snapshot)\n\ndef load_split(split):\n    df = pd.read_csv(DATA_ROOT / f"{split}.csv", low_memory=False)[["filename", "class", "image_path", "split", "is_frame", "source_class"]]\n    if split in (EVAL_SPLIT, TEST_SPLIT):\n        df = df[~df["class"].isin(EVAL_DROP)].reset_index(drop=True)\n    unknown = set(df["class"]) - set(PROBLEM_CLASSES) - set(CLEAN_CLASSES)\n    assert not unknown, unknown\n    df["is_problem"] = df["class"].isin(PROBLEM_CLASSES)\n    df["is_obstacle"] = df["class"].eq("Obstacle")\n    return df\n\ndef read_images(df):\n    def read(path):\n        with Image.open(DATA_ROOT / path) as im:\n            assert im.size == (224, 224), f\'Unexpected image geometry: {path}\'\n            return np.array(im.convert(\'RGB\'), copy=True)\n    with ThreadPoolExecutor(16) as pool:\n        return torch.from_numpy(np.stack(list(pool.map(read, df.image_path)))).permute(0, 3, 1, 2)  # NHWC to NCHW\n\nSPLITS = sorted({split for plan in TRAIN_PLANS.values() for split, _ in plan} | {EVAL_SPLIT, TEST_SPLIT})\ndata = {s: load_split(s) for s in SPLITS}\nfor s, df in data.items():\n    print(f"{s}: {len(df)} images, {(~df.is_problem).sum()} clean, {df.is_problem.sum()} problem, "\n          f"{df.is_obstacle.sum()} Obstacle")\nassert len(data[EVAL_SPLIT]) == 382 and len(data[TEST_SPLIT]) == 380\nassert all(df.image_path.is_unique for df in data.values())\nsave_json(RUN_DIR / \'data_manifest.json\', {s:dict(images=len(df), class_counts=df[\'class\'].value_counts().to_dict(), sha256=EXPECTED_MANIFEST_SHA256[s]) for s,df in data.items()})\n\n# CELL 10\nfrom torch import nn\nfrom transformers import AutoModel\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nuse_amp = device == "cuda"\nAMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\nMEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)  # ImageNet stats\nSTD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)\n\n@torch.inference_mode()\ndef embed(model, df):\n    """CLS token and mean patch token per image, both cached so pooling can be swept without re-embedding."""\n    cls, mean = [], []\n    for i in range(0, len(df), CACHE_BATCH_SIZE):\n        batch = (read_images(df.iloc[i:i + CACHE_BATCH_SIZE]).to(device).float() / 255 - MEAN) / STD\n        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):\n            hs = model(pixel_values=batch).last_hidden_state\n        cls.append(hs[:, 0].float())\n        mean.append(hs[:, 1 + model.config.num_register_tokens:].mean(1).float())\n    return dict(cls=torch.cat(cls), mean=torch.cat(mean))\n\nfeatures = {}\nfor name in {c["base_model"] for c in CONFIGS}:\n    revision_file = RUN_DIR / \'backbone_revision.json\'\n    if revision_file.exists():\n        model_revision = json.loads(revision_file.read_text())[\'revision\']\n    else:\n        model_revision = HfApi(token=HF_TOKEN).model_info(name).sha\n        save_json(revision_file, dict(model=name, revision=model_revision))\n    model = AutoModel.from_pretrained(name, revision=model_revision, token=HF_TOKEN).to(device).eval().requires_grad_(False)\n    t0 = time.perf_counter()\n    features[name] = {s: embed(model, df) for s, df in data.items()}\n    print(f"{name}: {sum(map(len, data.values())) / (time.perf_counter() - t0):.0f} images/s")\n    del model\nif device == "cuda":\n    torch.cuda.empty_cache()\n\ndef pooled(cfg, split):\n    f = features[cfg["base_model"]][split]\n    return torch.cat([f[p] for p in cfg["pooling"].split("+")], 1)\n\ndef targets(split):\n    df = data[split]\n    return (torch.tensor(df.is_problem.values, device=device).float(),\n            torch.tensor(df.is_obstacle.values, device=device).float())\nsave_json(RUN_DIR / \'feature_extraction.json\', dict(precision=str(AMP_DTYPE), batch_size=CACHE_BATCH_SIZE, backbone_revision=model_revision, pooling=\'cls\', backbone_frozen=True))\n\n# CELL 12\nfrom sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,\n                             precision_recall_fscore_support)\n\nCLEAN, OBSTACLE, SURFACE = "clean", "Obstacle", "SurfaceProblem"\n\ndef evaluate(df, p_problem, p_obstacle, k=K, stage_1_threshold=THRESHOLD, stage_2_threshold = THRESHOLD):\n    y = df.is_problem.values\n    hits = y[np.argsort(-p_problem, kind="stable")[:k]].sum() # Number of problems detected in the top K most confident predictions\n    pred_problem = p_problem >= stage_1_threshold\n    m = {f"recall@{k}": hits / y.sum(), f"precision@{k}": hits / k, f"recall@{k}_cap": min(k, y.sum()) / y.sum(),\n         "s1_accuracy": accuracy_score(y, pred_problem), "s1_balanced_accuracy": balanced_accuracy_score(y, pred_problem)}\n\n    true_obs, pred_obs = df.is_obstacle.values[y], p_obstacle[y] >= stage_2_threshold\n    p, r, _, _ = precision_recall_fscore_support(true_obs, pred_obs, labels=[True, False], zero_division=0)\n    m |= {"s2_accuracy": accuracy_score(true_obs, pred_obs), "s2_balanced_accuracy": balanced_accuracy_score(true_obs, pred_obs),\n          "s2_macro_precision": p.mean(), "s2_macro_recall": r.mean(), "s2_obstacle_precision": p[0],\n          "s2_obstacle_recall": r[0], "s2_surface_precision": p[1], "s2_surface_recall": r[1]}\n\n    names = [CLEAN, OBSTACLE, SURFACE]\n    true3 = np.where(~y, CLEAN, np.where(df.is_obstacle.values, OBSTACLE, SURFACE))\n    pred3 = np.where(~pred_problem, CLEAN, np.where(p_obstacle >= stage_2_threshold, OBSTACLE, SURFACE))\n    confusion = dict(\n        s1_cm=confusion_matrix(y, pred_problem, labels=[False, True]),\n        s2_cm=confusion_matrix(true_obs, pred_obs, labels=[True, False]),\n        pipeline_cm=confusion_matrix(true3, pred3, labels=names),\n        s1_by_class=pd.crosstab(df["source_class"], pred_problem).reindex(columns=[False, True], fill_value=0).values,\n        s1_by_class_names=sorted(df["source_class"].unique()))\n    return m, confusion\n\n# CELL 13\ndef batches(n, steps, batch_size, generator):\n    """Fixed number of batches, drawn without replacement and reshuffled once the split runs out."""\n    perm, i = torch.randperm(n, generator=generator), 0\n    for _ in range(steps):\n        if i + batch_size > n:\n            perm, i = torch.randperm(n, generator=generator), 0\n        yield perm[i:i + batch_size].to(device)\n        i += batch_size\n\ndef hierarchical_loss(logits, y_problem, y_obstacle):\n    stage1 = nn.functional.binary_cross_entropy_with_logits(logits[:, 0], y_problem)\n    mask = y_problem > 0\n    if not mask.any():\n        return stage1, torch.zeros_like(stage1)\n    stage2 = nn.functional.binary_cross_entropy_with_logits(logits[mask, 1], y_obstacle[mask])\n    return stage1, stage2\n\n@torch.inference_mode()\ndef predict(head, x):\n    p = torch.sigmoid(head(x)).cpu().numpy()\n    return p[:, 0], p[:, 1]\n\ndef run(cfg):\n    torch.manual_seed(cfg["seed"])\n    generator = torch.Generator().manual_seed(cfg["seed"])\n    head = nn.Linear(pooled(cfg, EVAL_SPLIT).shape[1], 2).to(device)\n    opt = torch.optim.Adam(head.parameters(), lr=cfg["lr"], weight_decay = cfg["weight_decay"])\n    val_x, val_df, val_y = pooled(cfg, EVAL_SPLIT), data[EVAL_SPLIT], targets(EVAL_SPLIT)\n    plan = TRAIN_PLANS[cfg["train"]]\n    total, step, rows, checkpoints, curve = sum(n for _, n in plan), 0, [], {}, []\n    for split, steps in plan:\n        x, (y_problem, y_obstacle) = pooled(cfg, split), targets(split)\n        for idx in batches(len(x), steps, cfg["batch_size"], generator):\n            s1_loss, s2_loss = hierarchical_loss(head(x[idx]), y_problem[idx], y_obstacle[idx])\n            loss = s1_loss + s2_loss\n            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()\n            curve.append(torch.stack([s1_loss, s2_loss]).detach())\n            step += 1\n            if step % cfg["eval_every"] == 0 or step == total:\n                m, _ = evaluate(val_df, *predict(head, val_x))\n                with torch.inference_mode():\n                    s1_val_loss, s2_val_loss = hierarchical_loss(head(val_x), *val_y)\n                rows.append({**cfg, "step": step, "phase": split, "s1_train_loss": s1_loss.item(), "s2_train_loss": s2_loss.item(), "train_loss": loss.item(),\n                             "s1_val_loss": s1_val_loss.item(), "s2_val_loss": s2_val_loss.item(), **m})\n                checkpoints[step] = {k: v.cpu().clone() for k, v in head.state_dict().items()}\n    return rows, checkpoints, torch.stack(curve).cpu().numpy()\n\n# CELL 15\nhistory, heads, curves = [], {}, []\nsave_json(RUN_DIR / \'status.json\', dict(status=\'training\', models_expected=len(CONFIGS), models_completed=0))\nt0 = time.perf_counter()\nfor i, cfg in enumerate(CONFIGS):\n    key = trial_id(cfg)\n    folder = RUN_DIR / \'trials\' / key\n    folder.mkdir(parents=True, exist_ok=True)\n    artifact = folder / \'complete.pt\'\n    try:\n        if artifact.is_file():\n            saved = torch.load(artifact, map_location=\'cpu\', weights_only=False)  # our own Drive artifact\n            assert saved[\'cfg\'] == cfg and saved[\'trial_id\'] == key\n            rows, checkpoints, curve = saved[\'rows\'], saved[\'checkpoints\'], saved[\'curve\']\n            action = \'reused\'\n        else:\n            rows, checkpoints, curve = run(cfg)\n            save_torch(artifact, dict(trial_id=key, cfg=cfg, rows=rows, checkpoints=checkpoints, curve=curve))\n            action = \'trained\'\n        pd.DataFrame(rows).to_csv(folder / \'metrics.csv\', index=False)\n        save_json(folder / \'identity.json\', dict(trial_id=key, config=cfg, train_plan=TRAIN_PLANS[cfg[\'train\']]))\n        history += [{**row, \'run\': key} for row in rows]\n        heads[key] = dict(cfg=cfg, checkpoints=checkpoints)\n        curve_frame = pd.DataFrame(curve, columns=[\'stage1\',\'stage2\']).assign(step=np.arange(1,len(curve)+1), run=key, seed=cfg[\'seed\'], **{k:cfg[k] for k in GROUP})\n        curve_frame.to_csv(folder / \'losses.csv\', index=False)\n        curves.append(curve_frame)\n        message = f"[{i+1}/{len(CONFIGS)}] {action} {key}: {cfg[\'train\']}, lr={cfg[\'lr\']:g}, wd={cfg[\'weight_decay\']:g}, seed={cfg[\'seed\']} | val precision@{K}={rows[-1][f\'precision@{K}\']:.4f}, subtype BA={rows[-1][\'s2_balanced_accuracy\']:.4f}"\n        print(message, flush=True)\n        with (RUN_DIR / \'sweep.log\').open(\'a\') as log: log.write(message + \'\\n\')\n        save_json(RUN_DIR / \'status.json\', dict(status=\'training\', models_expected=len(CONFIGS), models_completed=len(heads)))\n    except BaseException as error:\n        save_json(RUN_DIR / \'failure.json\', dict(trial_id=key, config=cfg, error_type=type(error).__name__, error=str(error)))\n        raise\nhistory = pd.DataFrame(history)\nlosses = pd.concat(curves, ignore_index=True)\nhistory.to_csv(RUN_DIR / \'metrics.csv\', index=False)\nlosses.to_csv(RUN_DIR / \'losses.csv\', index=False)\nsave_torch(RUN_DIR / \'heads.pt\', heads)\nassert len(heads) == len(CONFIGS) == 84\nprint(\'Saved all 84 trials:\', RUN_DIR, \'| sweep seconds:\', time.perf_counter()-t0)\n\n# CELL 17\npd.set_option("display.width", 200, "display.max_columns", 40)\nMETRICS = [f"recall@{K}", f"precision@{K}", "s1_accuracy", "s1_balanced_accuracy", "s2_accuracy", "s2_balanced_accuracy",\n           "s2_macro_precision", "s2_macro_recall", "s2_obstacle_precision", "s2_obstacle_recall",\n           "s2_surface_precision", "s2_surface_recall"]\nfinal = history[history.groupby(GROUP + ["seed"]).step.transform("max") == history.step]\nprint(f"recall@{K} cap: {final[f\'recall@{K}_cap\'].iloc[0]:.3f}")\nsummary = final.groupby(GROUP)[METRICS].agg(["mean", "std"]).round(3)\nsummary.to_csv(RUN_DIR / "summary.csv")\nfor m in METRICS:\n    print(f"\\n{m}, mean (std) over seeds")\n    t = summary[m]\n    print((t["mean"].map("{:.3f}".format) + " (" + t["std"].map("{:.3f}".format) + ")").unstack("lr").to_string())\n\n# CELL 18\nimport matplotlib.pyplot as plt\n\nSMOOTH = 25\nlrs = sorted(losses.lr.unique())\ncolors = dict(zip(lrs, plt.cm.Blues(np.linspace(.35, 1, len(lrs)))))\npanels = list(zip(losses.groupby(COMPARE), history.groupby(COMPARE)))\nfig, axes = plt.subplots(2, len(panels), figsize=(6 * len(panels), 7), sharex="col", sharey="row", squeeze=False)\nfor row, (stage, val_loss, title) in enumerate([("stage1", "s1_val_loss", "Stage 1 BCE"), ("stage2", "s2_val_loss", "Stage 2 BCE")]):\n    for col, ((values, panel), (_, evals)) in enumerate(panels):\n        ax = axes[row, col]\n        for (lr, seed), g in panel.groupby(["lr", "seed"]):\n            ax.plot(g.step, g[stage].rolling(SMOOTH, min_periods=1).mean(), color=colors[lr], lw=1.2,\n                    label=f"lr={lr:g}" if seed == panel.seed.min() else None)\n        for lr, g in evals.groupby("lr"):\n            g = g.groupby("step")[val_loss].mean()\n            ax.plot(g.index, g.values, color=colors[lr], lw=1.8, marker="o", ms=4)\n        steps = 0\n        for split, n in TRAIN_PLANS[panel.train.iloc[0]][:-1]:\n            steps += n\n            ax.axvline(steps, color="0.5", ls="--", lw=1)\n        ax.set_yscale("log")\n        ax.grid(alpha=.2)\n        ax.set_title(f"{\', \'.join(map(str, values))}: {title}")\n        ax.set_xlabel("Step")\n        ax.set_ylabel("Loss")\naxes[0, -1].plot([], [], color="0.3", lw=1.8, marker="o", ms=4, label="val, seed mean (one per lr)")\naxes[0, -1].legend(frameon=False)\nfig.tight_layout()\nfig.savefig(RUN_DIR / "loss_curves.png", dpi=150)\nplt.show()\n\n# CELL 20\ndef score(entry, split, step=None):\n    cfg, ckpts = entry["cfg"], entry["checkpoints"]\n    x = pooled(cfg, split)\n    head = nn.Linear(x.shape[1], 2).to(device)\n    head.load_state_dict(ckpts[step or max(ckpts)])\n    return predict(head, x)\n\ndef threshold_table(split):\n    rows, confusions = [], {}\n    for run_id, entry in heads.items():\n        p_problem, p_obstacle = score(entry, split)\n        for t1, t2 in product(THRESHOLDS, THRESHOLDS):\n            m, c = evaluate(data[split], p_problem, p_obstacle, stage_1_threshold=t1, stage_2_threshold=t2)\n            rows.append({**entry["cfg"], "run": run_id, "thresh1": t1, "thresh2": t2, **m})\n            confusions[run_id, t1, t2] = c\n    return pd.DataFrame(rows), confusions\n\ndef run_predictions(split, runs):\n    def one(run_id):\n        cfg = heads[run_id]["cfg"]\n        p_problem, p_obstacle = score(heads[run_id], split)\n        return data[split][["filename", "image_path", "split", "class", "source_class"]].assign(\n            run=run_id, seed=cfg["seed"], p_problem=p_problem, p_obstacle=p_obstacle, **{k: cfg[k] for k in GROUP})\n    return pd.concat([one(run_id) for run_id in runs])\n\ndef select(v):\n    """Learning rate by ranking, then each stage\'s threshold by its own metric, all on val means over seeds."""\n    lr = v.groupby("lr")[f"precision@{K}"].mean().idxmax()\n    v = v[v.lr == lr]\n    return lr, v.groupby("thresh1").s1_balanced_accuracy.mean().idxmax(), v.groupby("thresh2").s2_balanced_accuracy.mean().idxmax()\n\nval, val_confusions = threshold_table(EVAL_SPLIT)\nval.to_csv(RUN_DIR / "val_thresholds.csv", index=False)\nrun_predictions(EVAL_SPLIT, heads).to_csv(RUN_DIR / "val_predictions.csv", index=False)\nval.groupby(GROUP + ["thresh1", "thresh2"])[METRICS].agg(["mean", "std"]).round(3).to_csv(RUN_DIR / "threshold_summary.csv")\nprint("Mean stage 1 balanced accuracy over seeds, by stage 1 threshold")\nprint(val.groupby(GROUP + ["thresh1"]).s1_balanced_accuracy.mean().unstack("thresh1").round(3).to_string())\nprint("\\nMean stage 2 balanced accuracy over seeds, by stage 2 threshold")\nprint(val.groupby(GROUP + ["thresh2"]).s2_balanced_accuracy.mean().unstack("thresh2").round(3).to_string())\n\n\ndef make_choice(choice, model_entries):\n    selected = [key for key,entry in model_entries.items() if all(entry[\'cfg\'][k] == choice[k] for k in GROUP)]\n    selected.sort(key=lambda key:model_entries[key][\'cfg\'][\'seed\'])\n    assert len(selected) == 3 and [model_entries[key][\'cfg\'][\'seed\'] for key in selected] == [0,1,2], f\'Incomplete seed group: {choice}\'\n    return {**choice, \'runs\': selected}\n\ndef validate_choice(choice, model_entries):\n    expected = make_choice(choice, model_entries)[\'runs\']\n    assert choice[\'runs\'] == expected, \'Stale selection: run IDs do not match the named configuration.\'\n    for key in expected:\n        assert trial_id(model_entries[key][\'cfg\']) == key, \'Configuration ID mismatch.\'\n\nCHOSEN = []\nfor values, v in val.groupby(COMPARE):\n    lr, t1, t2 = select(v)\n    choice = dict(zip(COMPARE, values)) | dict(lr=float(lr), thresh1=float(t1), thresh2=float(t2),\n        choice_id=\'validation_\' + \'_\'.join(map(str,values)), selection_kind=\'fresh_validation_selection\')\n    CHOSEN.append(make_choice(choice, heads))\nHISTORICAL = [make_choice(c, heads) for c in REPORTED_CHOICES]\nTEST_CHOICES = HISTORICAL + CHOSEN\nfor choice in TEST_CHOICES: validate_choice(choice, heads)\npd.DataFrame(CHOSEN).to_csv(RUN_DIR / \'chosen.csv\', index=False)\npd.DataFrame(HISTORICAL).to_csv(RUN_DIR / \'reported_choices.csv\', index=False)\nsave_json(RUN_DIR / \'chosen.json\', CHOSEN)\nsave_json(RUN_DIR / \'reported_choices.json\', HISTORICAL)\nsave_json(RUN_DIR / \'selection_audit.json\', dict(rule=\'Within each train/weight-decay group: learning rate by mean validation precision@46; stage thresholds by their own mean validation balanced accuracies; lower values break ties.\',\n    validation_choices=CHOSEN, fixed_report_targets=HISTORICAL, test_used_for_selection=False))\nsave_json(RUN_DIR / \'val_confusions.json\', [{\'run\':key[0], \'thresh1\':key[1], \'thresh2\':key[2], **value} for key,value in val_confusions.items()])\nsave_json(RUN_DIR / \'status.json\', dict(status=\'validation_complete\', models_completed=len(heads)))\nprint(\'Fresh validation choices:\')\ndisplay(pd.DataFrame(CHOSEN))\nprint(\'Fixed historical report targets:\')\ndisplay(pd.DataFrame(HISTORICAL))\n\n# CELL 21\ndef show_confusion(chosen, confusions):\n    keys = [(run_id, chosen["thresh1"], chosen["thresh2"]) for run_id in chosen["runs"]]\n    total = lambda name: sum(confusions[k][name] for k in keys)\n    print(f"\\n=== {({k: v for k, v in chosen.items() if k != \'runs\'})}, {len(keys)} seeds ===")\n    print(pd.DataFrame(total("s1_cm"), ["clean", "problem"], ["pred clean", "pred problem"]).to_string())\n    print(pd.DataFrame(total("s1_by_class"), confusions[keys[0]]["s1_by_class_names"], ["pred clean", "pred problem"]).to_string())\n    print(pd.DataFrame(total("s2_cm"), ["Obstacle", "SurfaceProblem"], ["pred Obstacle", "pred SurfaceProblem"]).to_string())\n    print(pd.DataFrame(total("pipeline_cm"), [CLEAN, OBSTACLE, SURFACE], ["pred " + n for n in (CLEAN, OBSTACLE, SURFACE)]).to_string())\n\nfor chosen in TEST_CHOICES:\n    show_confusion(chosen, val_confusions)\n\n# CELL 23\nif RUN_TEST:\n    # Reload the saved selection records and verify them against the current sweep.\n    TEST_CHOICES = json.loads((RUN_DIR / \'reported_choices.json\').read_text()) + json.loads((RUN_DIR / \'chosen.json\').read_text())\n    test_rows, test_confusions, class_rows = [], [], []\n    for chosen in TEST_CHOICES:\n        validate_choice(chosen, heads)\n        for key in chosen[\'runs\']:\n            cfg = heads[key][\'cfg\']\n            p_problem, p_obstacle = score(heads[key], TEST_SPLIT)\n            m, cm = evaluate(data[TEST_SPLIT], p_problem, p_obstacle, stage_1_threshold=chosen[\'thresh1\'], stage_2_threshold=chosen[\'thresh2\'])\n            context = dict(choice_id=chosen[\'choice_id\'], selection_kind=chosen[\'selection_kind\'], run=key, **cfg, thresh1=chosen[\'thresh1\'], thresh2=chosen[\'thresh2\'])\n            test_rows.append({**context, **m})\n            test_confusions.append({**context, **cm})\n            for task, names, counts in [(\'hazard\',[\'clean\',\'hazard\'],cm[\'s1_cm\']),(\'subtype\',[\'Obstacle\',\'SurfaceProblem\'],cm[\'s2_cm\']),(\'pipeline\',[\'clean\',\'Obstacle\',\'SurfaceProblem\'],cm[\'pipeline_cm\'])]:\n                for j,label in enumerate(names):\n                    tp=int(counts[j,j]); support=int(counts[j].sum()); predicted=int(counts[:,j].sum())\n                    class_rows.append({**context, \'task\':task, \'label\':label, \'support\':support,\n                        \'precision\':tp/predicted if predicted else 0., \'recall\':tp/support if support else 0.,\n                        \'f1\':2*tp/(support+predicted) if support+predicted else 0.})\n    test = pd.DataFrame(test_rows)\n    test.to_csv(RUN_DIR / \'test_metrics.csv\', index=False)\n    pd.DataFrame(class_rows).to_csv(RUN_DIR / \'test_per_class_metrics.csv\', index=False)\n    save_json(RUN_DIR / \'test_confusions.json\', test_confusions)\n    test_predictions = run_predictions(TEST_SPLIT, sorted(test.run.unique()))\n    assert len(test_predictions) == 380 * test.run.nunique()\n    test_predictions.to_csv(RUN_DIR / \'test_predictions.csv\', index=False)\n    for kind, label in [(\'historical_report_reproduction\',\'reported\'),(\'fresh_validation_selection\',\'validation_selected\')]:\n        subset=test[test.selection_kind == kind]\n        subset.to_csv(RUN_DIR / f\'{label}_test_metrics.csv\', index=False)\n        subset.groupby([\'choice_id\']+GROUP+[\'thresh1\',\'thresh2\'])[METRICS].agg([\'mean\',\'std\']).to_csv(RUN_DIR / f\'{label}_test_summary.csv\')\n    test.groupby([\'choice_id\',\'selection_kind\']+GROUP+[\'thresh1\',\'thresh2\'])[METRICS].agg([\'mean\',\'std\']).to_csv(RUN_DIR / \'test_summary.csv\')\n    display(test.groupby([\'choice_id\',\'selection_kind\'])[[f\'precision@{K}\',\'s1_balanced_accuracy\',\'s2_balanced_accuracy\']].agg([\'mean\',\'std\']))\n    save_json(RUN_DIR / \'status.json\', dict(status=\'complete\', models_completed=len(heads), test_choices=len(TEST_CHOICES), test_prediction_rows=len(test_predictions)))\n    print(\'Finished. Preserve this entire Drive folder:\', RUN_DIR)\nelse:\n    print(\'Validation artifacts saved. Test evaluation disabled by RUN_TEST=False.\')'
run_config = dict(version='dino-report-rerun-v1', source_commit='180c69581a0571c7dcbf8ab9db3fa86f58fa9693',
    base=BASE, sweeps=SWEEPS, configs=CONFIGS, train_plans=TRAIN_PLANS, data_repo=DATA_REPO, data_tar=DATA_TAR,
    k=K, thresholds=THRESHOLDS, eval_split=EVAL_SPLIT, test_split=TEST_SPLIT,
    reported_choices=REPORTED_CHOICES, expected_manifest_sha256=EXPECTED_MANIFEST_SHA256,
    experiment_code_sha256=hashlib.sha256(EXPERIMENT_SOURCE.encode()).hexdigest())
RUN_DIR = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else DRIVE_ROOT / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:6])
if RESUME_RUN_DIR:
    assert (RUN_DIR / 'config.json').is_file(), 'Resume folder must contain config.json.'
    assert json.loads((RUN_DIR / 'config.json').read_text()) == json.loads(json.dumps(run_config)), 'Resume configuration/code changed; start a new run.'
else:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    save_json(RUN_DIR / 'config.json', run_config)
(RUN_DIR / 'experiment_source.py').write_text(EXPERIMENT_SOURCE)
probe = RUN_DIR / '.write_probe'
probe.write_text('ok'); assert probe.read_text() == 'ok'; probe.unlink()
save_json(RUN_DIR / 'environment.json', dict(python=sys.version, gpu=torch.cuda.get_device_name(0),
    torch=torch.__version__, transformers=transformers.__version__, cuda=torch.version.cuda,
    packages={p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','huggingface-hub','Pillow']}))
save_json(RUN_DIR / 'status.json', dict(status='initialized', models_expected=len(CONFIGS)))
print('Persistent results:', RUN_DIR)

## Dataset and provenance
All four manifests must match the dataset used in the report. Only training splits contribute gradients; validation drives selection. The test split is only scored in the final cell.

In [ ]:
import tarfile, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image

import getpass
from huggingface_hub import HfApi
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face read token (dataset and DINOv3 access): ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required for the private dataset and gated DINOv3 model.')

if not all((DATA_ROOT / f'{s}.csv').is_file() for s in EXPECTED_MANIFEST_SHA256):
    t0 = time.perf_counter()
    revision = HfApi(token=HF_TOKEN).dataset_info(DATA_REPO).sha
    tar = hf_hub_download(DATA_REPO, DATA_TAR, repo_type='dataset', revision=revision, token=HF_TOKEN)
    with tarfile.open(tar) as archive:
        archive.extractall(DATA_ROOT.parent, filter='data')
    save_json(RUN_DIR / 'download.json', dict(repo=DATA_REPO, revision=revision, archive=DATA_TAR,
        archive_sha256=hashlib.file_digest(open(tar, 'rb'), 'sha256').hexdigest()))
    print(f'downloaded and extracted in {time.perf_counter() - t0:.0f}s')

for split, expected in EXPECTED_MANIFEST_SHA256.items():
    manifest = DATA_ROOT / f'{split}.csv'
    assert hashlib.sha256(manifest.read_bytes()).hexdigest() == expected, f'Dataset drift: {split}.csv does not match the report dataset.'
    snapshot = RUN_DIR / 'manifests' / manifest.name
    snapshot.parent.mkdir(exist_ok=True)
    shutil.copy2(manifest, snapshot)

def load_split(split):
    df = pd.read_csv(DATA_ROOT / f"{split}.csv", low_memory=False)[["filename", "class", "image_path", "split", "is_frame", "source_class"]]
    if split in (EVAL_SPLIT, TEST_SPLIT):
        df = df[~df["class"].isin(EVAL_DROP)].reset_index(drop=True)
    unknown = set(df["class"]) - set(PROBLEM_CLASSES) - set(CLEAN_CLASSES)
    assert not unknown, unknown
    df["is_problem"] = df["class"].isin(PROBLEM_CLASSES)
    df["is_obstacle"] = df["class"].eq("Obstacle")
    return df

def read_images(df):
    def read(path):
        with Image.open(DATA_ROOT / path) as im:
            assert im.size == (224, 224), f'Unexpected image geometry: {path}'
            return np.array(im.convert('RGB'), copy=True)
    with ThreadPoolExecutor(16) as pool:
        return torch.from_numpy(np.stack(list(pool.map(read, df.image_path)))).permute(0, 3, 1, 2)  # NHWC to NCHW

SPLITS = sorted({split for plan in TRAIN_PLANS.values() for split, _ in plan} | {EVAL_SPLIT, TEST_SPLIT})
data = {s: load_split(s) for s in SPLITS}
for s, df in data.items():
    print(f"{s}: {len(df)} images, {(~df.is_problem).sum()} clean, {df.is_problem.sum()} problem, "
          f"{df.is_obstacle.sum()} Obstacle")
assert len(data[EVAL_SPLIT]) == 382 and len(data[TEST_SPLIT]) == 380
assert all(df.image_path.is_unique for df in data.values())
save_json(RUN_DIR / 'data_manifest.json', {s:dict(images=len(df), class_counts=df['class'].value_counts().to_dict(), sha256=EXPECTED_MANIFEST_SHA256[s]) for s,df in data.items()})

## Frozen DINOv3 features
The backbone is frozen. BF16 is used on supported GPUs (including the original A100); T4 uses FP16. The model revision and precision are recorded. Feature extraction is repeated after reconnecting; completed head-training trials are reused from Drive.

In [ ]:
from torch import nn
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)  # ImageNet stats
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.inference_mode()
def embed(model, df):
    """CLS token and mean patch token per image, both cached so pooling can be swept without re-embedding."""
    cls, mean = [], []
    for i in range(0, len(df), CACHE_BATCH_SIZE):
        batch = (read_images(df.iloc[i:i + CACHE_BATCH_SIZE]).to(device).float() / 255 - MEAN) / STD
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
            hs = model(pixel_values=batch).last_hidden_state
        cls.append(hs[:, 0].float())
        mean.append(hs[:, 1 + model.config.num_register_tokens:].mean(1).float())
    return dict(cls=torch.cat(cls), mean=torch.cat(mean))

features = {}
for name in {c["base_model"] for c in CONFIGS}:
    revision_file = RUN_DIR / 'backbone_revision.json'
    if revision_file.exists():
        model_revision = json.loads(revision_file.read_text())['revision']
    else:
        model_revision = HfApi(token=HF_TOKEN).model_info(name).sha
        save_json(revision_file, dict(model=name, revision=model_revision))
    model = AutoModel.from_pretrained(name, revision=model_revision, token=HF_TOKEN).to(device).eval().requires_grad_(False)
    t0 = time.perf_counter()
    features[name] = {s: embed(model, df) for s, df in data.items()}
    print(f"{name}: {sum(map(len, data.values())) / (time.perf_counter() - t0):.0f} images/s")
    del model
if device == "cuda":
    torch.cuda.empty_cache()

def pooled(cfg, split):
    f = features[cfg["base_model"]][split]
    return torch.cat([f[p] for p in cfg["pooling"].split("+")], 1)

def targets(split):
    df = data[split]
    return (torch.tensor(df.is_problem.values, device=device).float(),
            torch.tensor(df.is_obstacle.values, device=device).float())
save_json(RUN_DIR / 'feature_extraction.json', dict(precision=str(AMP_DTYPE), batch_size=CACHE_BATCH_SIZE, backbone_revision=model_revision, pooling='cls', backbone_frozen=True))

## Metrics and training
Precision@46 = true hazards among the 46 highest hazard scores / 46. Subtype balanced accuracy averages Obstacle and SurfaceProblem recall over all true hazards, independently of stage 1. Training remains the equal sum of two unweighted BCE losses.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

CLEAN, OBSTACLE, SURFACE = "clean", "Obstacle", "SurfaceProblem"

def evaluate(df, p_problem, p_obstacle, k=K, stage_1_threshold=THRESHOLD, stage_2_threshold = THRESHOLD):
    y = df.is_problem.values
    hits = y[np.argsort(-p_problem, kind="stable")[:k]].sum() # Number of problems detected in the top K most confident predictions
    pred_problem = p_problem >= stage_1_threshold
    m = {f"recall@{k}": hits / y.sum(), f"precision@{k}": hits / k, f"recall@{k}_cap": min(k, y.sum()) / y.sum(),
         "s1_accuracy": accuracy_score(y, pred_problem), "s1_balanced_accuracy": balanced_accuracy_score(y, pred_problem)}

    true_obs, pred_obs = df.is_obstacle.values[y], p_obstacle[y] >= stage_2_threshold
    p, r, _, _ = precision_recall_fscore_support(true_obs, pred_obs, labels=[True, False], zero_division=0)
    m |= {"s2_accuracy": accuracy_score(true_obs, pred_obs), "s2_balanced_accuracy": balanced_accuracy_score(true_obs, pred_obs),
          "s2_macro_precision": p.mean(), "s2_macro_recall": r.mean(), "s2_obstacle_precision": p[0],
          "s2_obstacle_recall": r[0], "s2_surface_precision": p[1], "s2_surface_recall": r[1]}

    names = [CLEAN, OBSTACLE, SURFACE]
    true3 = np.where(~y, CLEAN, np.where(df.is_obstacle.values, OBSTACLE, SURFACE))
    pred3 = np.where(~pred_problem, CLEAN, np.where(p_obstacle >= stage_2_threshold, OBSTACLE, SURFACE))
    confusion = dict(
        s1_cm=confusion_matrix(y, pred_problem, labels=[False, True]),
        s2_cm=confusion_matrix(true_obs, pred_obs, labels=[True, False]),
        pipeline_cm=confusion_matrix(true3, pred3, labels=names),
        s1_by_class=pd.crosstab(df["source_class"], pred_problem).reindex(columns=[False, True], fill_value=0).values,
        s1_by_class_names=sorted(df["source_class"].unique()))
    return m, confusion

In [ ]:
def batches(n, steps, batch_size, generator):
    """Fixed number of batches, drawn without replacement and reshuffled once the split runs out."""
    perm, i = torch.randperm(n, generator=generator), 0
    for _ in range(steps):
        if i + batch_size > n:
            perm, i = torch.randperm(n, generator=generator), 0
        yield perm[i:i + batch_size].to(device)
        i += batch_size

def hierarchical_loss(logits, y_problem, y_obstacle):
    stage1 = nn.functional.binary_cross_entropy_with_logits(logits[:, 0], y_problem)
    mask = y_problem > 0
    if not mask.any():
        return stage1, torch.zeros_like(stage1)
    stage2 = nn.functional.binary_cross_entropy_with_logits(logits[mask, 1], y_obstacle[mask])
    return stage1, stage2

@torch.inference_mode()
def predict(head, x):
    p = torch.sigmoid(head(x)).cpu().numpy()
    return p[:, 0], p[:, 1]

def run(cfg):
    torch.manual_seed(cfg["seed"])
    generator = torch.Generator().manual_seed(cfg["seed"])
    head = nn.Linear(pooled(cfg, EVAL_SPLIT).shape[1], 2).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=cfg["lr"], weight_decay = cfg["weight_decay"])
    val_x, val_df, val_y = pooled(cfg, EVAL_SPLIT), data[EVAL_SPLIT], targets(EVAL_SPLIT)
    plan = TRAIN_PLANS[cfg["train"]]
    total, step, rows, checkpoints, curve = sum(n for _, n in plan), 0, [], {}, []
    for split, steps in plan:
        x, (y_problem, y_obstacle) = pooled(cfg, split), targets(split)
        for idx in batches(len(x), steps, cfg["batch_size"], generator):
            s1_loss, s2_loss = hierarchical_loss(head(x[idx]), y_problem[idx], y_obstacle[idx])
            loss = s1_loss + s2_loss
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            curve.append(torch.stack([s1_loss, s2_loss]).detach())
            step += 1
            if step % cfg["eval_every"] == 0 or step == total:
                m, _ = evaluate(val_df, *predict(head, val_x))
                with torch.inference_mode():
                    s1_val_loss, s2_val_loss = hierarchical_loss(head(val_x), *val_y)
                rows.append({**cfg, "step": step, "phase": split, "s1_train_loss": s1_loss.item(), "s2_train_loss": s2_loss.item(), "train_loss": loss.item(),
                             "s1_val_loss": s1_val_loss.item(), "s2_val_loss": s2_val_loss.item(), **m})
                checkpoints[step] = {k: v.cpu().clone() for k, v in head.state_dict().items()}
    return rows, checkpoints, torch.stack(curve).cpu().numpy()

## Full sweep, with a saved result after every completed trial
Stable IDs derive from the complete configuration. Interrupted trials restart; completed trials are skipped. Checkpoints, metrics, and losses are saved per trial before proceeding.

In [ ]:
history, heads, curves = [], {}, []
save_json(RUN_DIR / 'status.json', dict(status='training', models_expected=len(CONFIGS), models_completed=0))
t0 = time.perf_counter()
for i, cfg in enumerate(CONFIGS):
    key = trial_id(cfg)
    folder = RUN_DIR / 'trials' / key
    folder.mkdir(parents=True, exist_ok=True)
    artifact = folder / 'complete.pt'
    try:
        if artifact.is_file():
            saved = torch.load(artifact, map_location='cpu', weights_only=False)  # our own Drive artifact
            assert saved['cfg'] == cfg and saved['trial_id'] == key
            rows, checkpoints, curve = saved['rows'], saved['checkpoints'], saved['curve']
            action = 'reused'
        else:
            rows, checkpoints, curve = run(cfg)
            save_torch(artifact, dict(trial_id=key, cfg=cfg, rows=rows, checkpoints=checkpoints, curve=curve))
            action = 'trained'
        pd.DataFrame(rows).to_csv(folder / 'metrics.csv', index=False)
        save_json(folder / 'identity.json', dict(trial_id=key, config=cfg, train_plan=TRAIN_PLANS[cfg['train']]))
        history += [{**row, 'run': key} for row in rows]
        heads[key] = dict(cfg=cfg, checkpoints=checkpoints)
        curve_frame = pd.DataFrame(curve, columns=['stage1','stage2']).assign(step=np.arange(1,len(curve)+1), run=key, seed=cfg['seed'], **{k:cfg[k] for k in GROUP})
        curve_frame.to_csv(folder / 'losses.csv', index=False)
        curves.append(curve_frame)
        message = f"[{i+1}/{len(CONFIGS)}] {action} {key}: {cfg['train']}, lr={cfg['lr']:g}, wd={cfg['weight_decay']:g}, seed={cfg['seed']} | val precision@{K}={rows[-1][f'precision@{K}']:.4f}, subtype BA={rows[-1]['s2_balanced_accuracy']:.4f}"
        print(message, flush=True)
        with (RUN_DIR / 'sweep.log').open('a') as log: log.write(message + '\n')
        save_json(RUN_DIR / 'status.json', dict(status='training', models_expected=len(CONFIGS), models_completed=len(heads)))
    except BaseException as error:
        save_json(RUN_DIR / 'failure.json', dict(trial_id=key, config=cfg, error_type=type(error).__name__, error=str(error)))
        raise
history = pd.DataFrame(history)
losses = pd.concat(curves, ignore_index=True)
history.to_csv(RUN_DIR / 'metrics.csv', index=False)
losses.to_csv(RUN_DIR / 'losses.csv', index=False)
save_torch(RUN_DIR / 'heads.pt', heads)
assert len(heads) == len(CONFIGS) == 84
print('Saved all 84 trials:', RUN_DIR, '| sweep seconds:', time.perf_counter()-t0)

## Validation sweep summaries

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 40)
METRICS = [f"recall@{K}", f"precision@{K}", "s1_accuracy", "s1_balanced_accuracy", "s2_accuracy", "s2_balanced_accuracy",
           "s2_macro_precision", "s2_macro_recall", "s2_obstacle_precision", "s2_obstacle_recall",
           "s2_surface_precision", "s2_surface_recall"]
final = history[history.groupby(GROUP + ["seed"]).step.transform("max") == history.step]
print(f"recall@{K} cap: {final[f'recall@{K}_cap'].iloc[0]:.3f}")
summary = final.groupby(GROUP)[METRICS].agg(["mean", "std"]).round(3)
summary.to_csv(RUN_DIR / "summary.csv")
for m in METRICS:
    print(f"\n{m}, mean (std) over seeds")
    t = summary[m]
    print((t["mean"].map("{:.3f}".format) + " (" + t["std"].map("{:.3f}".format) + ")").unstack("lr").to_string())

In [ ]:
import matplotlib.pyplot as plt

SMOOTH = 25
lrs = sorted(losses.lr.unique())
colors = dict(zip(lrs, plt.cm.Blues(np.linspace(.35, 1, len(lrs)))))
panels = list(zip(losses.groupby(COMPARE), history.groupby(COMPARE)))
fig, axes = plt.subplots(2, len(panels), figsize=(6 * len(panels), 7), sharex="col", sharey="row", squeeze=False)
for row, (stage, val_loss, title) in enumerate([("stage1", "s1_val_loss", "Stage 1 BCE"), ("stage2", "s2_val_loss", "Stage 2 BCE")]):
    for col, ((values, panel), (_, evals)) in enumerate(panels):
        ax = axes[row, col]
        for (lr, seed), g in panel.groupby(["lr", "seed"]):
            ax.plot(g.step, g[stage].rolling(SMOOTH, min_periods=1).mean(), color=colors[lr], lw=1.2,
                    label=f"lr={lr:g}" if seed == panel.seed.min() else None)
        for lr, g in evals.groupby("lr"):
            g = g.groupby("step")[val_loss].mean()
            ax.plot(g.index, g.values, color=colors[lr], lw=1.8, marker="o", ms=4)
        steps = 0
        for split, n in TRAIN_PLANS[panel.train.iloc[0]][:-1]:
            steps += n
            ax.axvline(steps, color="0.5", ls="--", lw=1)
        ax.set_yscale("log")
        ax.grid(alpha=.2)
        ax.set_title(f"{', '.join(map(str, values))}: {title}")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
axes[0, -1].plot([], [], color="0.3", lw=1.8, marker="o", ms=4, label="val, seed mean (one per lr)")
axes[0, -1].legend(frameon=False)
fig.tight_layout()
fig.savefig(RUN_DIR / "loss_curves.png", dpi=150)
plt.show()

## Recompute selection from the current validation predictions
`CHOSEN` is rebuilt here every time. The original rule selects a learning rate by precision@46 inside each training-plan/weight-decay group, then selects each decision threshold by its own balanced accuracy. Ties use the lower value. This is not a separate learning-rate search optimized for subtype accuracy.

The two fixed historical report targets are also recorded, under different names. Their hyperparameters are never described as fresh validation winners.

In [ ]:
def score(entry, split, step=None):
    cfg, ckpts = entry["cfg"], entry["checkpoints"]
    x = pooled(cfg, split)
    head = nn.Linear(x.shape[1], 2).to(device)
    head.load_state_dict(ckpts[step or max(ckpts)])
    return predict(head, x)

def threshold_table(split):
    rows, confusions = [], {}
    for run_id, entry in heads.items():
        p_problem, p_obstacle = score(entry, split)
        for t1, t2 in product(THRESHOLDS, THRESHOLDS):
            m, c = evaluate(data[split], p_problem, p_obstacle, stage_1_threshold=t1, stage_2_threshold=t2)
            rows.append({**entry["cfg"], "run": run_id, "thresh1": t1, "thresh2": t2, **m})
            confusions[run_id, t1, t2] = c
    return pd.DataFrame(rows), confusions

def run_predictions(split, runs):
    def one(run_id):
        cfg = heads[run_id]["cfg"]
        p_problem, p_obstacle = score(heads[run_id], split)
        return data[split][["filename", "image_path", "split", "class", "source_class"]].assign(
            run=run_id, seed=cfg["seed"], p_problem=p_problem, p_obstacle=p_obstacle, **{k: cfg[k] for k in GROUP})
    return pd.concat([one(run_id) for run_id in runs])

def select(v):
    """Learning rate by ranking, then each stage's threshold by its own metric, all on val means over seeds."""
    lr = v.groupby("lr")[f"precision@{K}"].mean().idxmax()
    v = v[v.lr == lr]
    return lr, v.groupby("thresh1").s1_balanced_accuracy.mean().idxmax(), v.groupby("thresh2").s2_balanced_accuracy.mean().idxmax()

val, val_confusions = threshold_table(EVAL_SPLIT)
val.to_csv(RUN_DIR / "val_thresholds.csv", index=False)
run_predictions(EVAL_SPLIT, heads).to_csv(RUN_DIR / "val_predictions.csv", index=False)
val.groupby(GROUP + ["thresh1", "thresh2"])[METRICS].agg(["mean", "std"]).round(3).to_csv(RUN_DIR / "threshold_summary.csv")
print("Mean stage 1 balanced accuracy over seeds, by stage 1 threshold")
print(val.groupby(GROUP + ["thresh1"]).s1_balanced_accuracy.mean().unstack("thresh1").round(3).to_string())
print("\nMean stage 2 balanced accuracy over seeds, by stage 2 threshold")
print(val.groupby(GROUP + ["thresh2"]).s2_balanced_accuracy.mean().unstack("thresh2").round(3).to_string())


def make_choice(choice, model_entries):
    selected = [key for key,entry in model_entries.items() if all(entry['cfg'][k] == choice[k] for k in GROUP)]
    selected.sort(key=lambda key:model_entries[key]['cfg']['seed'])
    assert len(selected) == 3 and [model_entries[key]['cfg']['seed'] for key in selected] == [0,1,2], f'Incomplete seed group: {choice}'
    return {**choice, 'runs': selected}

def validate_choice(choice, model_entries):
    expected = make_choice(choice, model_entries)['runs']
    assert choice['runs'] == expected, 'Stale selection: run IDs do not match the named configuration.'
    for key in expected:
        assert trial_id(model_entries[key]['cfg']) == key, 'Configuration ID mismatch.'

CHOSEN = []
for values, v in val.groupby(COMPARE):
    lr, t1, t2 = select(v)
    choice = dict(zip(COMPARE, values)) | dict(lr=float(lr), thresh1=float(t1), thresh2=float(t2),
        choice_id='validation_' + '_'.join(map(str,values)), selection_kind='fresh_validation_selection')
    CHOSEN.append(make_choice(choice, heads))
HISTORICAL = [make_choice(c, heads) for c in REPORTED_CHOICES]
TEST_CHOICES = HISTORICAL + CHOSEN
for choice in TEST_CHOICES: validate_choice(choice, heads)
pd.DataFrame(CHOSEN).to_csv(RUN_DIR / 'chosen.csv', index=False)
pd.DataFrame(HISTORICAL).to_csv(RUN_DIR / 'reported_choices.csv', index=False)
save_json(RUN_DIR / 'chosen.json', CHOSEN)
save_json(RUN_DIR / 'reported_choices.json', HISTORICAL)
save_json(RUN_DIR / 'selection_audit.json', dict(rule='Within each train/weight-decay group: learning rate by mean validation precision@46; stage thresholds by their own mean validation balanced accuracies; lower values break ties.',
    validation_choices=CHOSEN, fixed_report_targets=HISTORICAL, test_used_for_selection=False))
save_json(RUN_DIR / 'val_confusions.json', [{'run':key[0], 'thresh1':key[1], 'thresh2':key[2], **value} for key,value in val_confusions.items()])
save_json(RUN_DIR / 'status.json', dict(status='validation_complete', models_completed=len(heads)))
print('Fresh validation choices:')
display(pd.DataFrame(CHOSEN))
print('Fixed historical report targets:')
display(pd.DataFrame(HISTORICAL))

In [ ]:
def show_confusion(chosen, confusions):
    keys = [(run_id, chosen["thresh1"], chosen["thresh2"]) for run_id in chosen["runs"]]
    total = lambda name: sum(confusions[k][name] for k in keys)
    print(f"\n=== {({k: v for k, v in chosen.items() if k != 'runs'})}, {len(keys)} seeds ===")
    print(pd.DataFrame(total("s1_cm"), ["clean", "problem"], ["pred clean", "pred problem"]).to_string())
    print(pd.DataFrame(total("s1_by_class"), confusions[keys[0]]["s1_by_class_names"], ["pred clean", "pred problem"]).to_string())
    print(pd.DataFrame(total("s2_cm"), ["Obstacle", "SurfaceProblem"], ["pred Obstacle", "pred SurfaceProblem"]).to_string())
    print(pd.DataFrame(total("pipeline_cm"), [CLEAN, OBSTACLE, SURFACE], ["pred " + n for n in (CLEAN, OBSTACLE, SURFACE)]).to_string())

for chosen in TEST_CHOICES:
    show_confusion(chosen, val_confusions)

## Test evaluation and complete per-class records
Both fixed historical targets and freshly selected configurations are evaluated here. The files identify the selection source. Every image retains its path, true label, source label, model configuration, seed, and both probabilities. Confusion matrices are saved per seed before aggregation. No test score affects selection.

In [ ]:
if RUN_TEST:
    # Reload the saved selection records and verify them against the current sweep.
    TEST_CHOICES = json.loads((RUN_DIR / 'reported_choices.json').read_text()) + json.loads((RUN_DIR / 'chosen.json').read_text())
    test_rows, test_confusions, class_rows = [], [], []
    for chosen in TEST_CHOICES:
        validate_choice(chosen, heads)
        for key in chosen['runs']:
            cfg = heads[key]['cfg']
            p_problem, p_obstacle = score(heads[key], TEST_SPLIT)
            m, cm = evaluate(data[TEST_SPLIT], p_problem, p_obstacle, stage_1_threshold=chosen['thresh1'], stage_2_threshold=chosen['thresh2'])
            context = dict(choice_id=chosen['choice_id'], selection_kind=chosen['selection_kind'], run=key, **cfg, thresh1=chosen['thresh1'], thresh2=chosen['thresh2'])
            test_rows.append({**context, **m})
            test_confusions.append({**context, **cm})
            for task, names, counts in [('hazard',['clean','hazard'],cm['s1_cm']),('subtype',['Obstacle','SurfaceProblem'],cm['s2_cm']),('pipeline',['clean','Obstacle','SurfaceProblem'],cm['pipeline_cm'])]:
                for j,label in enumerate(names):
                    tp=int(counts[j,j]); support=int(counts[j].sum()); predicted=int(counts[:,j].sum())
                    class_rows.append({**context, 'task':task, 'label':label, 'support':support,
                        'precision':tp/predicted if predicted else 0., 'recall':tp/support if support else 0.,
                        'f1':2*tp/(support+predicted) if support+predicted else 0.})
    test = pd.DataFrame(test_rows)
    test.to_csv(RUN_DIR / 'test_metrics.csv', index=False)
    pd.DataFrame(class_rows).to_csv(RUN_DIR / 'test_per_class_metrics.csv', index=False)
    save_json(RUN_DIR / 'test_confusions.json', test_confusions)
    test_predictions = run_predictions(TEST_SPLIT, sorted(test.run.unique()))
    assert len(test_predictions) == 380 * test.run.nunique()
    test_predictions.to_csv(RUN_DIR / 'test_predictions.csv', index=False)
    for kind, label in [('historical_report_reproduction','reported'),('fresh_validation_selection','validation_selected')]:
        subset=test[test.selection_kind == kind]
        subset.to_csv(RUN_DIR / f'{label}_test_metrics.csv', index=False)
        subset.groupby(['choice_id']+GROUP+['thresh1','thresh2'])[METRICS].agg(['mean','std']).to_csv(RUN_DIR / f'{label}_test_summary.csv')
    test.groupby(['choice_id','selection_kind']+GROUP+['thresh1','thresh2'])[METRICS].agg(['mean','std']).to_csv(RUN_DIR / 'test_summary.csv')
    display(test.groupby(['choice_id','selection_kind'])[[f'precision@{K}','s1_balanced_accuracy','s2_balanced_accuracy']].agg(['mean','std']))
    save_json(RUN_DIR / 'status.json', dict(status='complete', models_completed=len(heads), test_choices=len(TEST_CHOICES), test_prediction_rows=len(test_predictions)))
    print('Finished. Preserve this entire Drive folder:', RUN_DIR)
else:
    print('Validation artifacts saved. Test evaluation disabled by RUN_TEST=False.')

## Files to share after completion
The complete Drive run folder is the best export. At minimum include `config.json`, `chosen.csv`, `chosen.json`, `reported_choices.json`, `selection_audit.json`, `test_metrics.csv`, `test_predictions.csv`, `test_summary.csv`, `test_per_class_metrics.csv`, `test_confusions.json`, `val_predictions.csv`, `val_thresholds.csv`, `data_manifest.json`, and `status.json`. `heads.pt` and `trials/` preserve checkpoints for later reuse.

The current rerun can differ numerically from the historical experiment because hardware, precision, package versions, and the previously unpinned backbone revision can differ. Their actual values are recorded in `environment.json`, `feature_extraction.json`, and `backbone_revision.json`.